In [23]:
# Out-of-fold prediction analysis: which human texts were flagged as AI?
# All predictions are out-of-fold, so no data leaks

import textwrap
import pandas as pd
import json
from pathlib import Path

WRAP_WIDTH = 90  # wrap printed text to this many characters

OOF_PATH = Path("roberta-finetuned-pooled/cv/oof_predictions.csv")
DATA_PATH = Path("/mimer/NOBACKUP/groups/naiss2025-22-1187/coherence-driven-humans/data/post-processing/cleaned_outputs.json")

df = pd.read_csv(OOF_PATH)
print(f"Total OOF predictions: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head()

Total OOF predictions: 480
Columns: ['fold', 'story_id', 'seed', 'model_type', 'true_label', 'predicted_label', 'prob_human', 'prob_ai']


,fold,story_id,seed,model_type,true_label,predicted_label,prob_human,prob_ai
0,fold-1,3230,42,qwen3vl,1,1,0.000956,0.999044
1,fold-1,11260,42,qwen3vl,1,1,0.000915,0.999085
2,fold-1,4173,42,qwen3vl,1,1,0.001263,0.998737
3,fold-1,11678,42,qwen3vl,1,1,0.000828,0.999172
4,fold-1,13683,42,qwen3vl,1,1,0.000869,0.999131


In [24]:
# Overview: prediction counts and detection rates per model type
summary = (
    df.groupby("model_type")
    .agg(
        n=("predicted_label", "count"),
        n_predicted_ai=("predicted_label", "sum"),
        mean_prob_ai=("prob_ai", "mean"),
    )
)
summary

,n,n_predicted_ai,mean_prob_ai
model_type,,,
claude45,60,60,0.999439
gpt4o,60,60,0.999400
human,180,7,0.038423
internvl3,60,60,0.998903
llama4scout,60,60,0.999495
qwen3vl,60,60,0.999486


In [25]:
# Human texts flagged as AI-generated (false positives)
human_df = df[df["model_type"] == "human"].copy()
flagged = human_df[human_df["predicted_label"] == 1].sort_values("prob_ai", ascending=False)

print(f"Human texts flagged as AI: {len(flagged)} / {len(human_df)} "
      f"({100 * len(flagged) / len(human_df):.1f}%)")
print()

# How many unique story_ids were flagged, and how many seeds per story_id
flagged_counts = flagged.groupby("story_id")["seed"].count().sort_values(ascending=False)
print(f"Unique story_ids flagged: {len(flagged_counts)} / {human_df['story_id'].nunique()}")
print()
for sid, count in flagged_counts.items():
    seeds = sorted(flagged[flagged["story_id"] == sid]["seed"].tolist())
    print(f"  story_id {sid}: {count}/3 seeds flagged (seeds {seeds})")

print()
flagged[["fold", "story_id", "seed", "prob_human", "prob_ai"]]

Human texts flagged as AI: 7 / 180 (3.9%)

Unique story_ids flagged: 7 / 60

  story_id 1214: 1/3 seeds flagged (seeds [42])
  story_id 3761: 1/3 seeds flagged (seeds [42])
  story_id 3891: 1/3 seeds flagged (seeds [43])
  story_id 5047: 1/3 seeds flagged (seeds [42])
  story_id 5540: 1/3 seeds flagged (seeds [42])
  story_id 7195: 1/3 seeds flagged (seeds [43])
  story_id 10499: 1/3 seeds flagged (seeds [42])



,fold,story_id,seed,prob_human,prob_ai
364,fold-4,5540,42,0.000481,0.999519
185,fold-2,10499,42,0.000929,0.999071
369,fold-4,7195,43,0.001157,0.998843
359,fold-4,3761,42,0.005070,0.994930
161,fold-2,1214,42,0.006358,0.993642
362,fold-4,5047,42,0.046091,0.953909
263,fold-3,3891,43,0.281955,0.718045


In [28]:
# Load actual texts of flagged stories
with open(DATA_PATH) as f:
    raw = json.load(f)

text_lookup = {
    (r["story_id"], r["seed"], r["model_type"]): r["cleaned_model_output"]
    for r in raw
}

In [30]:
# Display the full text of each flagged human story
print("HUMAN TEXTS MISCLASSIFIED AS AI")

for _, row in flagged.iterrows():
    key = (row["story_id"], row["seed"], "human")
    text = text_lookup.get(key, "[text not found]")
    text_clean = text.replace("[SEP] ", "")

    print(f"\n{'─' * WRAP_WIDTH}")
    print(f"story_id={row['story_id']}  seed={row['seed']}  "
          f"fold={row['fold']}  P(AI)={row['prob_ai']:.4f}")
    print(f"{'─' * WRAP_WIDTH}")
    print(textwrap.fill(text_clean, width=WRAP_WIDTH))
    print()

HUMAN TEXTS MISCLASSIFIED AS AI

──────────────────────────────────────────────────────────────────────────────────────────
story_id=5540  seed=42  fold=fold-4  P(AI)=0.9995
──────────────────────────────────────────────────────────────────────────────────────────
Three men sit together in a small diner, talking quietly over coffee. They lean toward
each other as if discussing something serious. The mood feels tense but controlled, and
each man listens carefully while trying not to draw attention. A man in a winter coat
stands at a counter, speaking to someone across from him. He looks uneasy, holding a piece
of paper while the person listens. The light from the window makes the scene feel cold and
quiet. Outside the diner, two men stand talking near the door. One wears a red coat while
the other keeps his hands in his pockets against the cold. Their breath shows in the
winter air as they speak quietly, looking uneasy. The man in the tan coat stands near the
road, his face tense as he 

In [31]:
# Are the same story_ids flagged across seeds?
# Check whether certain visual sequences are consistently hard for the classifier.
flagged_story_ids = flagged["story_id"].unique()

related = human_df[human_df["story_id"].isin(flagged_story_ids)].sort_values(
    ["story_id", "seed"]
)
related[["story_id", "seed", "fold", "predicted_label", "prob_human", "prob_ai"]]

,story_id,seed,fold,predicted_label,prob_human,prob_ai
161,1214,42,fold-2,1,0.006358,0.993642
159,1214,43,fold-2,0,0.999800,0.000200
160,1214,44,fold-2,0,0.999798,0.000202
359,3761,42,fold-4,1,0.005070,0.994930
357,3761,43,fold-4,0,0.999746,0.000254
358,3761,44,fold-4,0,0.999745,0.000255
262,3891,42,fold-3,0,0.999625,0.000375
263,3891,43,fold-3,1,0.281955,0.718045
261,3891,44,fold-3,0,0.999623,0.000377
362,5047,42,fold-4,1,0.046091,0.953909


In [32]:
# Word count comparison: flagged vs correctly classified human texts
human_df = human_df.copy()
human_df["text"] = human_df.apply(
    lambda r: text_lookup.get((r["story_id"], r["seed"], "human"), ""), axis=1
)
human_df["word_count"] = human_df["text"].str.replace(r"\[SEP\] ", "", regex=True).str.split().str.len()

print("Word count stats — correctly classified as human:")
print(human_df[human_df["predicted_label"] == 0]["word_count"].describe().round(1))
print()
print("Word count stats — misclassified as AI:")
print(human_df[human_df["predicted_label"] == 1]["word_count"].describe().round(1))

Word count stats — correctly classified as human:
count    173.0
mean     162.8
std       81.3
min       85.0
25%      111.0
50%      137.0
75%      179.0
max      487.0
Name: word_count, dtype: float64

Word count stats — misclassified as AI:
count      7.0
mean     154.1
std       60.9
min       88.0
25%       98.5
50%      148.0
75%      210.5
max      225.0
Name: word_count, dtype: float64
